In [14]:
import pandas as pd
import datetime
from espn_nfl_scrape import nfl_week_sunday
from api_calls import getNextWeeksSpreads, extractNextWeeksSpreads, add_to_db
from api_calls import get_db, mongoConn


In [15]:
df_scores = pd.read_excel('data/nfl.xlsx')
df_scores['diff'] = df_scores['Home Score'] - df_scores['Away Score']
df_home = df_scores[['Date','Home Team','diff','Away Team']].rename(columns={'Home Team': 'Team','Away Team': 'Opponenet'})
df_away = df_scores[['Date','Away Team','diff','Home Team']].rename(columns={'Away Team': 'Team', 'Home Team': 'Opponenet'})
df_away['diff'] = -df_away['diff']
df_scores_all = pd.concat([df_home, df_away]).reset_index()
df_scores_all['join_date'] = pd.to_datetime(df_scores_all['Date']).dt.normalize()

# df_scores_all['Date'] = pd.to_datetime(df_scores_all['Date']).dt.tz_convert('America/New_York').dt.date
df_scores_all[df_scores_all['Team']=='Baltimore Ravens']

,index,Date,Team,diff,Opponenet,join_date
9,9,2025-01-12 00:00:00,Baltimore Ravens,14,Pittsburgh Steelers,2025-01-12
28,28,2025-01-04 00:00:00,Baltimore Ravens,25,Cleveland Browns,2025-01-04
58,58,2024-12-21 00:00:00,Baltimore Ravens,17,Pittsburgh Steelers,2024-12-21
92,92,2024-12-01 00:00:00,Baltimore Ravens,-5,Philadelphia Eagles,2024-12-01
146,146,2024-11-07 00:00:00,Baltimore Ravens,1,Cincinnati Bengals,2024-11-07
...,...,...,...,...,...,...
10149,5003,2006-11-12 00:00:00,Baltimore Ravens,1,Tennessee Titans,2006-11-12
10184,5038,2006-10-29 00:00:00,Baltimore Ravens,13,New Orleans Saints,2006-10-29
10218,5072,2006-10-09 00:00:00,Baltimore Ravens,-10,Denver Broncos,2006-10-09
10257,5111,2006-09-24 00:00:00,Baltimore Ravens,1,Cleveland Browns,2006-09-24


In [16]:
from espn_nfl_scrape import nfl_week_sunday
year = 2023
week = 1

nfl_sunday_date = nfl_week_sunday(year)
wednesday_date = nfl_sunday_date[0] - datetime.timedelta(days=3)
tuesday_date = nfl_sunday_date[-1] + datetime.timedelta(days=2)
print(nfl_sunday_date[0], wednesday_date, tuesday_date)

2023-09-10 2023-09-07 2024-01-09


In [17]:
datesUTC = [wednesday_date,tuesday_date]
api_pay_type = 'paid'
spreaddata = getNextWeeksSpreads(datesUTC,api_pay_type)

check spreaddata
number of games in spreaddata before date filter = 272
number of games in spreaddata after date filter = 272
Remaining requests 18749.0
Used requests 1251


In [18]:
def extract_first_spreads(spreaddata,bet='spreads'):
    rows = []
    for g in spreaddata:
        date_iso = g.get("commence_time")           # e.g. '2024-09-06T00:20:00Z'
        bms = g.get("bookmakers", [])
        if not bms:
            continue
        # use FIRST bookmaker only
        markets = bms[0].get("markets", [])
        spread_mkt = next((m for m in markets if m.get("key") == bet), None)
        if not spread_mkt:
            continue
        for o in spread_mkt.get("outcomes", []):    # two rows: one per team
            rows.append({
                "team": o.get("name"),
                "date": date_iso,                   # keep full ISO; or date_iso[:10] for just YYYY-MM-DD
                bet: o.get("point"),
            })
    return pd.DataFrame(rows)

df_spreads = extract_first_spreads(spreaddata,bet='spreads' )
df_spreads['join_date'] = (
    pd.to_datetime(df_spreads['date'], utc=True)          # -> UTC tz-aware
      .dt.tz_convert('America/New_York')                        # -> ET
      .dt.tz_localize(None)                                     # drop tz
      .dt.normalize()                                           # midnight
)
df_spreads

,team,date,spreads,join_date
0,Detroit Lions,2023-09-08T00:20:00Z,5.5,2023-09-07
1,Kansas City Chiefs,2023-09-08T00:20:00Z,-5.5,2023-09-07
2,Arizona Cardinals,2023-09-10T17:00:00Z,7.0,2023-09-10
3,Washington Commanders,2023-09-10T17:00:00Z,-7.0,2023-09-10
4,Atlanta Falcons,2023-09-10T17:00:00Z,-3.5,2023-09-10
...,...,...,...,...
539,New York Jets,2024-01-07T18:00:00Z,-1.0,2024-01-07
540,New York Giants,2024-01-07T18:00:00Z,3.0,2024-01-07
541,Philadelphia Eagles,2024-01-07T18:00:00Z,-3.0,2024-01-07
542,Los Angeles Rams,2024-01-07T21:00:00Z,7.0,2024-01-07


In [19]:
df_spreadscore = df_spreads.merge(df_scores_all, left_on=['team','join_date'], right_on=['Team','join_date'], how='left')
df_spreadscore['spreadscore'] = df_spreadscore['diff'] + df_spreadscore['spreads']
df_spreadscore[['team','date','spreads','diff','spreadscore']]

,team,date,spreads,diff,spreadscore
0,Detroit Lions,2023-09-08T00:20:00Z,5.5,1.0,6.5
1,Kansas City Chiefs,2023-09-08T00:20:00Z,-5.5,-1.0,-6.5
2,Arizona Cardinals,2023-09-10T17:00:00Z,7.0,-4.0,3.0
3,Washington Commanders,2023-09-10T17:00:00Z,-7.0,4.0,-3.0
4,Atlanta Falcons,2023-09-10T17:00:00Z,-3.5,14.0,10.5
...,...,...,...,...,...
539,New York Jets,2024-01-07T18:00:00Z,-1.0,14.0,13.0
540,New York Giants,2024-01-07T18:00:00Z,3.0,17.0,20.0
541,Philadelphia Eagles,2024-01-07T18:00:00Z,-3.0,-17.0,-20.0
542,Los Angeles Rams,2024-01-07T21:00:00Z,7.0,1.0,8.0


In [28]:
client = mongoConn()
dfbets = get_db(client, 'withTheSpread', 'bets')
dfseasonspreads = get_db(client, 'withTheSpread', 'season_spreads')
dfbets['Bet'].unique()
# dfbets['Year'].unique()
dfseasonspreads[dfseasonspreads['Year']==2025]

Pinged your deployment. You successfully connected to MongoDB!


,_id,Team,Week,opponent,spread,score,diff,Year,spreadscore,coverprob,predspread,order,coverprob_diff,predspread_diff
1184,68bf1a45fd43f4e9f651ef74,Dallas Cowboys,1,Philadelphia Eagles,8.5,NaN,NaN,2025,NaN,0.5,0.0,0.0,0.0,0.0
1185,68bf1a45fd43f4e9f651ef75,Philadelphia Eagles,1,Dallas Cowboys,-8.5,NaN,NaN,2025,NaN,0.5,0.0,0.0,0.0,0.0
1186,68bf1a45fd43f4e9f651ef76,Kansas City Chiefs,1,Los Angeles Chargers,-3.0,NaN,NaN,2025,NaN,0.5,0.0,1.0,0.0,0.0
1187,68bf1a45fd43f4e9f651ef77,Los Angeles Chargers,1,Kansas City Chiefs,3.0,NaN,NaN,2025,NaN,0.5,0.0,1.0,0.0,0.0
1188,68bf1a45fd43f4e9f651ef78,Arizona Cardinals,1,New Orleans Saints,-6.5,NaN,NaN,2025,NaN,0.5,0.0,2.0,0.0,0.0
1189,68bf1a45fd43f4e9f651ef79,New Orleans Saints,1,Arizona Cardinals,6.5,NaN,NaN,2025,NaN,0.5,0.0,2.0,0.0,0.0
1190,68bf1a45fd43f4e9f651ef7a,Atlanta Falcons,1,Tampa Bay Buccaneers,1.5,NaN,NaN,2025,NaN,0.5,0.0,3.0,0.0,0.0
1191,68bf1a45fd43f4e9f651ef7b,Tampa Bay Buccaneers,1,Atlanta Falcons,-1.5,NaN,NaN,2025,NaN,0.5,0.0,3.0,0.0,0.0
1192,68bf1a45fd43f4e9f651ef7c,Carolina Panthers,1,Jacksonville Jaguars,3.5,NaN,NaN,2025,NaN,0.5,0.0,4.0,0.0,0.0
1193,68bf1a45fd43f4e9f651ef7d,Jacksonville Jaguars,1,Carolina Panthers,-3.5,NaN,NaN,2025,NaN,0.5,0.0,4.0,0.0,0.0


In [23]:
teams = df_spreadscore['team'].unique()
df_spreadscore[['team','date','spreads','diff','spreadscore']]
df_totals = extract_first_spreads(spreaddata,bet='totals')
df_spreads = extract_first_spreads(spreaddata,bet='spreads')

dfs = []
for bet, df in zip(['spreadscore','totals','spreads'],[df_spreadscore,df_spreads,df_spreads]):
    temp_df = pd.DataFrame(index=range(1,19))
    rename_dct = {'spreadscore':'SpreadScore','totals':'OverUnder','spreads':'Spread'}
    for team in teams:
        s_team = df_spreadscore[df_spreadscore['team'] == team]['spreadscore']
        s_dates = df_spreadscore[df_spreadscore['team'] == team]['date']
        s_dates = df_spreadscore[df_spreadscore['team'] == team]['date']
        s_dates = pd.to_datetime(s_dates)
        s_weeks = ((s_dates - s_dates.iloc[0]).dt.days / 7).round()+1
        s_weeks.loc[s_weeks.diff()==0] = s_weeks.loc[s_weeks.diff()==0]+1
        assert s_dates.value_counts().max()==1
        s_team.index = s_weeks.values
        # team_spreads[team] = s_team
        temp_df = temp_df.merge(s_team.rename(team), left_index=True, right_index=True, how='left')
    # df_bets.columns = [str(i) for i in range(1,len(df_bets.columns)+2)]
    temp_df = temp_df.T
    temp_df.columns = [str(i) for i in range(1,len(temp_df.columns)+1)]
    temp_df['Bet'] = rename_dct[bet]
    dfs.append(temp_df)

df_bets = pd.concat(dfs)
df_bets['Year'] = year
df_bets['teamyearid'] = df_bets.index + '-' + df_bets['Year'].astype(str)
df_bets = df_bets.reset_index().rename(columns={'index':'Team'})
df_bets = df_bets[[col for col in dfbets.columns if col != '_id']]
df_bets





,Team,Year,teamyearid,Bet,1,2,3,4,5,6,...,9,10,11,12,13,14,15,16,17,18
0,Detroit Lions,2023,Detroit Lions-2023,SpreadScore,6.5,-8.5,9.0,12.5,13.0,10.5,...,NaN,6.0,0.0,-11.0,6.0,6.0,-16.5,NaN,7.0,2.5
1,Kansas City Chiefs,2023,Kansas City Chiefs-2023,SpreadScore,-6.5,5.0,21.5,1.0,3.0,4.0,...,1.5,NaN,NaN,-7.0,9.0,-13.5,-6.5,NaN,NaN,-15.5
2,Arizona Cardinals,2023,Arizona Cardinals-2023,SpreadScore,3.0,1.5,19.0,-8.5,-6.5,-12.5,...,-20.0,4.5,-2.5,-20.5,20.5,20.5,NaN,-9.0,-6.5,15.0
3,Washington Commanders,2023,Washington Commanders-2023,SpreadScore,-3.0,5.5,-29.5,4.0,-21.5,11.5,...,6.5,1.0,-10.5,-29.0,-26.5,-26.5,NaN,-7.0,3.5,-13.5
4,Atlanta Falcons,2023,Atlanta Falcons-2023,SpreadScore,10.5,-0.5,-9.0,-11.5,-1.5,-11.5,...,-2.0,-4.5,NaN,10.0,11.5,11.5,-7.0,-1.0,16.0,-18.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,Philadelphia Eagles,2023,Philadelphia Eagles-2023,Spread,1.0,-1.0,7.0,-4.0,2.5,-7.0,...,2.0,NaN,7.0,0.5,-25.5,-25.5,-19.0,NaN,0.5,-15.0
92,Dallas Cowboys,2023,Dallas Cowboys-2023,Spread,36.5,17.0,-19.0,30.5,-29.5,4.0,...,-2.0,26.5,19.5,29.0,2.5,2.5,19.0,-17.0,-0.5,-2.5
93,New York Giants,2023,New York Giants-2023,Spread,-36.5,-1.5,-12.0,-22.0,-10.5,2.0,...,-22.5,-26.5,10.5,1.5,NaN,NaN,-0.5,-15.5,-0.5,-4.0
94,Buffalo Bills,2023,Buffalo Bills-2023,Spread,-8.5,19.0,29.5,24.5,-8.5,-2.0,...,-5.0,-8.0,22.0,-0.5,NaN,NaN,6.5,17.0,1.0,-0.5


In [24]:
add_to_db(client,'withTheSpread','bets',df_bets)

In [27]:
sorted(df_bets['Team'].unique()) == sorted(dfbets['Team'].unique())

True

In [87]:
df_bets[dfbets['Year']==2024]

KeyError: "['Team', '18'] not in index"

In [24]:
dfseasonspreads

,_id,Team,Week,opponent,spread,score,diff,Year,spreadscore,coverprob,predspread,order,coverprob_diff,predspread_diff
0,65163843441cb69ddfdd1630,Cincinnati Bengals,3,Los Angeles Rams,-1.5,19.0,3.0,2023,1.5,0.507081,-1.633186,15.0,-0.015729,-0.114765
1,65163843441cb69ddfdd1629,Carolina Panthers,2,New Orleans Saints,3.0,17.0,-3.0,2023,0.0,0.468306,0.263147,NaN,0.066440,NaN
2,65163843441cb69ddfdd162a,Chicago Bears,2,Tampa Bay Buccaneers,2.5,17.0,-10.0,2023,-7.5,0.405467,-7.578706,NaN,-0.103738,NaN
3,65163843441cb69ddfdd1638,Kansas City Chiefs,3,Chicago Bears,-12.5,41.0,31.0,2023,18.5,0.440254,8.711520,12.0,-0.066827,15.995833
4,65163843441cb69ddfdd1665,Tennessee Titans,2,Los Angeles Chargers,2.5,27.0,3.0,2023,5.5,0.454472,-6.536362,NaN,0.115370,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1209,68bf1a45fd43f4e9f651ef8d,Green Bay Packers,1,Detroit Lions,-2.5,NaN,NaN,2025,NaN,0.500000,0.000000,12.0,0.000000,0.000000
1210,68bf1a45fd43f4e9f651ef8e,Houston Texans,1,Los Angeles Rams,3.0,NaN,NaN,2025,NaN,0.500000,0.000000,13.0,0.000000,0.000000
1211,68bf1a45fd43f4e9f651ef8f,Los Angeles Rams,1,Houston Texans,-3.0,NaN,NaN,2025,NaN,0.500000,0.000000,13.0,0.000000,0.000000
1212,68bf1a45fd43f4e9f651ef90,Baltimore Ravens,1,Buffalo Bills,-1.5,NaN,NaN,2025,NaN,0.500000,0.000000,14.0,0.000000,0.000000
